# Entrenamiento del clasificador de dominios (módulo DNS)

Este notebook entrena el modelo que usa `dns_sumidero.py` para detectar dominios
**maliciosos** (phishing y malware) y dominios **DGA** (generados por malware) que
**no están en ninguna lista de bloqueo**.

**Antes de empezar:** en el panel izquierdo de Colab (ícono de carpeta) sube el archivo
`caracteristicas.py`. Es el mismo que usa la pi; si no son iguales, el modelo no sirve.

Clases del modelo:

| Clase | Significado | Fuente de los ejemplos |
| --- | --- | --- |
| 0 | Legítimo | Lista Tranco (sitios más visitados) y, opcional, tu `consultas.csv` |
| 1 | Malicioso (phishing / malware) | OpenPhish y URLhaus |
| 2 | DGA | Generadores sintéticos tipo DGA (ver nota en la celda 4) |

Ejecuta las celdas en orden con **Entorno de ejecución → Ejecutar todas**.

In [ ]:
# 1. Misma versión de scikit-learn que en la pi (si no coinciden, el modelo no carga)
!pip -q install scikit-learn==1.6.1 joblib==1.4.2

In [ ]:
# 2. Descargar los datos
import io, zipfile, urllib.request, random
import pandas as pd
from urllib.parse import urlparse

random.seed(42)

def descargar(url):
    peticion = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    return urllib.request.urlopen(peticion, timeout=60).read()

# Legítimos: lista Tranco (ranking de sitios más visitados)
z = zipfile.ZipFile(io.BytesIO(descargar("https://tranco-list.eu/top-1m.csv.zip")))
tranco = pd.read_csv(z.open(z.namelist()[0]), header=None, names=["rank", "dominio"])
print("Tranco:", len(tranco), "dominios")

# Maliciosos: OpenPhish (URLs de phishing) y URLhaus (hosts que distribuyen malware)
openphish = [urlparse(u.strip()).hostname for u in descargar("https://openphish.com/feed.txt").decode().splitlines() if u.strip()]
urlhaus = [l.split()[1] for l in descargar("https://urlhaus.abuse.ch/downloads/hostfile/").decode(errors="ignore").splitlines()
           if l and not l.startswith("#") and len(l.split()) >= 2]
print("OpenPhish:", len(openphish), "| URLhaus:", len(urlhaus))

In [ ]:
# 3. Limpiar los maliciosos
import ipaddress
from caracteristicas import partir

def es_ip(texto):
    try:
        ipaddress.ip_address(texto); return True
    except ValueError:
        return False

# Dominios muy populares (top 10.000): se usan para descartar falsos maliciosos
# (por ejemplo, phishing alojado en docs.google.com o github.io: el dominio en sí es legítimo)
top10k = set(tranco["dominio"].head(10000))

def dominio_registrable(d):
    sub, principal, tld = partir(d)
    return f"{principal}.{tld}"

maliciosos = set()
for h in openphish + urlhaus:
    if not h or es_ip(h):
        continue
    h = h.lower().strip(".")
    if dominio_registrable(h) in top10k:
        continue
    maliciosos.add(h)
maliciosos = sorted(maliciosos)
print("Maliciosos útiles:", len(maliciosos))

### Nota sobre los DGA

Los malware con DGA generan miles de dominios a partir de una fórmula. Aquí se generan
dominios **sintéticos** con cuatro estilos comunes (letras al azar, hexadecimal, letras y
números, y palabras de diccionario). Es suficiente para el prototipo; para una versión más
fuerte se pueden usar familias reales del repositorio `baderj/domain_generation_algorithms`
de GitHub o de DGArchive. Esto debe mencionarse como limitación en el artículo.

In [ ]:
# 4. Generar dominios tipo DGA
import string
TLDS_DGA = ["com", "net", "org", "info", "biz", "ru", "cn", "top", "xyz", "pw", "cc"]
PALABRAS = ("house world light water money power story night table river music paper party "
            "green black white right place point state great house small large north south "
            "smart cloud stone metal fresh quick happy sound field heart dream space trade").split()

def dga_letras():
    return "".join(random.choices(string.ascii_lowercase, k=random.randint(8, 22)))
def dga_hex():
    return "".join(random.choices("0123456789abcdef", k=random.randint(12, 32)))
def dga_mixto():
    return "".join(random.choices(string.ascii_lowercase + string.digits, k=random.randint(8, 20)))
def dga_diccionario():
    return "".join(random.sample(PALABRAS, random.randint(2, 3)))

generadores = [dga_letras, dga_hex, dga_mixto, dga_diccionario]
N_DGA = 40000
dga = [f"{random.choice(generadores)()}.{random.choice(TLDS_DGA)}" for _ in range(N_DGA)]
print(dga[:8])

In [ ]:
# 5. Armar el conjunto de datos balanceado
N_LEGIT = 60000
# Mezcla de sitios muy populares y menos populares
legitimos = list(tranco["dominio"].head(20000)) + random.sample(list(tranco["dominio"].iloc[20000:]), N_LEGIT - 20000)

# OPCIONAL: agregar dominios reales de tu red (incluyen subdominios como i.ytimg.com).
# Sube tu consultas.csv al panel de Colab y se usarán los PERMITIDOS.
import os
if os.path.exists("consultas.csv"):
    propios = pd.read_csv("consultas.csv")
    propios = propios.loc[propios["accion"] == "PERMITIDO", "dominio"].dropna().unique().tolist()
    legitimos += propios
    print("Dominios propios agregados:", len(propios))

datos = pd.DataFrame({
    "dominio": legitimos + maliciosos + dga,
    "clase": [0] * len(legitimos) + [1] * len(maliciosos) + [2] * len(dga),
}).drop_duplicates("dominio")
print(datos["clase"].value_counts().rename({0: "legítimo", 1: "malicioso", 2: "DGA"}))

In [ ]:
# 6. Extraer características y entrenar
from caracteristicas import extraer, COLUMNAS
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix

X = pd.DataFrame([extraer(d) for d in datos["dominio"]], columns=COLUMNAS)
y = datos["clase"].values
X_ent, X_prueba, y_ent, y_prueba = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

modelo = RandomForestClassifier(n_estimators=150, min_samples_leaf=2, class_weight="balanced",
                                n_jobs=-1, random_state=42)
modelo.fit(X_ent, y_ent)

pred = modelo.predict(X_prueba)
nombres = ["legítimo", "malicioso", "DGA"]
print(classification_report(y_prueba, pred, target_names=nombres, digits=4))
print("Matriz de confusión (filas = real, columnas = predicho):")
print(pd.DataFrame(confusion_matrix(y_prueba, pred), index=nombres, columns=nombres))

pliegues = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_cv = cross_val_score(modelo, X, y, cv=pliegues, scoring="f1_macro", n_jobs=-1)
print(f"\nF1 macro con validación cruzada de 5 particiones: {f1_cv.mean():.4f} ± {f1_cv.std():.4f}")

Guarda estas métricas (reporte, matriz de confusión y F1 de validación cruzada): son
**resultados para el artículo** (Tabla 6, efectividad del módulo DNS).

In [ ]:
# 7. Importancia de cada característica (sirve para la discusión del artículo)
importancia = pd.Series(modelo.feature_importances_, index=COLUMNAS).sort_values(ascending=False)
importancia.plot.barh(figsize=(7, 5), title="Importancia de las características").invert_yaxis()
importancia

In [ ]:
# 8. Guardar el modelo y la lista de dominios populares, y descargarlos
import joblib, sklearn, datetime
paquete = {
    "modelo": modelo,
    "columnas": COLUMNAS,
    "clases": nombres,
    "sklearn": sklearn.__version__,
    "fecha": datetime.date.today().isoformat(),
}
joblib.dump(paquete, "modelo_dominios.joblib", compress=3)
pd.Series(sorted(top10k)).to_csv("permitidos_top.txt", index=False, header=False)

from google.colab import files
files.download("modelo_dominios.joblib")
files.download("permitidos_top.txt")

In [ ]:
# 9. Prueba rápida con dominios de ejemplo
ejemplos = ["google.com", "umsa.bo", "bancounion.com.bo", "youtube.com",
            "paypal-login-verify-account.xyz", "secure-bcp-bo-acceso.top",
            "xjwqkzpltrmvbn.ru", "a3f9c2e1d7b8f6a4.info", "greenriverparty.com"]
probas = modelo.predict_proba(pd.DataFrame([extraer(d) for d in ejemplos], columns=COLUMNAS))
pd.DataFrame(probas, index=ejemplos, columns=nombres).round(3)